# Day 5 – Delta Lake Advanced: Step-by-Step Notes

## Topics Covered
* **Time travel (version history)**: Query previous versions of your Delta table for auditing, recovery, and reproducibility.
* **MERGE operations (upserts)**: Efficiently insert or update data, supporting incremental pipelines and change data capture.
* **OPTIMIZE & ZORDER**: Compact files and organize data for faster queries, especially on large tables.
* **VACUUM for cleanup**: Remove obsolete files to free up storage and maintain data hygiene.

## Steps and Reasoning

**Step 1: Implement incremental MERGE**
- Purpose: Efficiently update your Delta table with new or changed data using a single command.
- Command: `merge` matches records on key columns and upserts (updates/inserts) as needed.
- Reason: Ensures your table stays current without manual deduplication or overwrites.

**Step 2: Query historical versions (Time Travel)**
- Purpose: Access previous states of your data for auditing, debugging, or reproducing analyses.
- Command: Use `option("versionAsOf", ...)` or `option("timestampAsOf", ...)` to read older versions.
- Reason: Delta Lake’s transaction log enables safe, reliable time travel.

**Step 3: Optimize tables (Compaction & ZORDER)**
- Purpose: Improve query performance by reducing the number of files and clustering data by key columns.
- Command: `OPTIMIZE ... ZORDER BY (...)` compacts files and sorts data for efficient access.
- Reason: Large tables can become fragmented; optimization speeds up queries and reduces costs.

**Step 4: Clean old files (VACUUM)**
- Purpose: Remove files no longer referenced by the Delta log, freeing up space and keeping your data lake tidy.
- Command: `VACUUM ... RETAIN ... HOURS` deletes obsolete files after a retention period.
- Reason: Maintains storage hygiene and compliance with data retention policies.

---

Each step is designed to make your Delta Lake tables more reliable, performant, and manageable for advanced analytics and production workloads.

MERGE


In [0]:
# Step 1: Read CSV
oct_df = spark.read.csv(
    "/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv",
    header=True,
    inferSchema=True
)

# Step 2: Set Delta table name
managed_table_name = "ecommerce.events"

# Step 3: Save as Delta table (overwrite if exists)
oct_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(managed_table_name)

# Step 4: Import DeltaTable and Spark functions
from delta.tables import DeltaTable
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

# Step 5: Load Delta table
delta_table = DeltaTable.forName(spark, managed_table_name)

# Step 6: Deduplicate source to avoid multiple source row errors
window_spec = Window.partitionBy(
    "user_session",
    "event_time",
    "product_id",
    "event_type"
).orderBy(col("event_time").desc())

updates_df = (
    oct_df
    .withColumn("rn", row_number().over(window_spec))
    .filter(col("rn") == 1)
    .drop("rn")
)

# Step 7: Check count before merge
before_count = spark.read.table(managed_table_name).count()
print("Before merge count of rows:", before_count)

# Step 8: Perform MERGE
delta_table.alias("t").merge(
    updates_df.alias("s"),
    """
    t.user_session = s.user_session AND
    t.event_time = s.event_time AND
    t.product_id = s.product_id AND
    t.event_type = s.event_type
    """
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

# Step 9: Check count after merge
after_count = spark.read.table(managed_table_name).count()
print("After merge count of rows:", after_count)


QUERY VERSIONS

In [0]:
from delta.tables import DeltaTable

# Load table
delta_table = DeltaTable.forPath(spark, delta_path)

# a) Show Delta history
spark.sql(f"DESCRIBE HISTORY delta.`{delta_path}`").show(truncate=False)

# b) Query an old version by version number
version_0_df = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load(delta_path)

print("Number of rows in version 0:", version_0_df.count())

# c) Query as of a timestamp
from datetime import datetime

# Example: all rows as of Jan 1, 2026
yesterday_df = spark.read.format("delta") \
    .option("timestampAsOf", "2026-01-12 15:20:23") \
    .load(delta_path)

print("Number of rows as of timestamp:", yesterday_df.count())

OPTIMIZE TABLES (ZORDER)

In [0]:
# Optimize Delta table with ZORDER by columns often queried
spark.sql(f"OPTIMIZE delta.`{delta_path}` ZORDER BY (event_type, user_id)")

CLEAN OLD FILES (VACCUM)

In [0]:
# Remove files no longer needed by Delta Lake (default retention: 7 days)
spark.sql("VACUUM ecommerce.events RETAIN 168 HOURS")

# Delta Lake Advanced: Additional Notes and Examples

## Best Practices
* **MERGE Operations**: Always match on unique or primary key columns to avoid duplicate records. Validate your source DataFrame schema before merging.
* **Time Travel**: Use `DESCRIBE HISTORY <table>` to list available versions and commit timestamps before querying historical data.
* **OPTIMIZE & ZORDER**: Run OPTIMIZE periodically on large tables, especially after heavy ingest or update workloads. ZORDER by columns frequently used in WHERE clauses for best performance.
* **VACUUM**: Set retention periods according to your data governance policies. Never use very low retention unless you are sure no queries need older files.

## Practical Examples
**1. View Table History**
```python
spark.sql("DESCRIBE HISTORY ecommerce.events").show(truncate=False)
```

**2. Time Travel by Timestamp**
```python
# Find a valid timestamp from table history, then query
history = spark.sql("DESCRIBE HISTORY ecommerce.events").toPandas()
timestamp = history['timestamp'][0]  # Use a real timestamp from history
df_time = spark.read.format("delta").option("timestampAsOf", str(timestamp)).table("ecommerce.events")
display(df_time)
```

**3. MERGE with Real Data**
```python
# Example: Upsert new sales data
new_sales = spark.read.csv("/Volumes/workspace/ecommerce/new_sales.csv", header=True, inferSchema=True)
deltaTable.alias("t").merge(
    new_sales.alias("s"),
    "t.order_id = s.order_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()
```

**4. OPTIMIZE and ZORDER for Multiple Columns**
```python
spark.sql("OPTIMIZE ecommerce.events ZORDER BY (event_type, user_id, product_id)")
```

**5. VACUUM with Custom Retention**
```python
spark.sql("VACUUM ecommerce.events RETAIN 336 HOURS")  # Retain 14 days
```

## Troubleshooting
* If you get errors about missing versions or timestamps, always check table history first.
* If MERGE fails, verify that join columns exist in both source and target tables.
* If OPTIMIZE or VACUUM is slow, check for active queries or large numbers of small files.

---

These notes and examples will help you apply Delta Lake’s advanced features more effectively in real-world scenarios.